In [199]:
import pandas as pd
import numpy as np

import requests as req
from bs4 import BeautifulSoup as B, NavigableString, Tag, Comment, Doctype
import time


In [200]:
# links = []
# titles = []
# for i in range(1, 49):
#     url = f'https://news.tvbs.com.tw/news/searchresult/%E8%98%87%E8%8A%B1%E5%85%AC%E8%B7%AF/news/{i}'
#     resp = req.get(url)
#     if resp.status_code != 200 :
#         print('Error status_code')
#     else:
#         soup = B(resp.text, 'html.parser')
#         sites = soup.find_all('div', class_='list')
#         for s in sites:
#             ul_ = s.find('ul')
#             if not ul_:
#                 continue
#             for li_ in ul_.find_all('li'):
#                 a_tag = li_.find('a', href=True)
#                 h2_tag = li_.find('h2', class_='txt')
#                 if a_tag and h2_tag:
#                     link = a_tag['href']
#                     title = h2_tag.get_text(strip=True)
#                     links.append(link)
#                     titles.append(title)
    # time.sleep(1)

In [201]:
# data = pd.DataFrame({'Link': links, 'Title': titles})

In [202]:
# data.shape

In [203]:
# data.duplicated().sum()

In [204]:
# data.to_csv('tvbs_蘇花公路.csv', index=False)

In [205]:
data = pd.read_csv('tvbs_蘇花公路.csv')
data.head()

,Link,Title
0,https://news.tvbs.com.tw/local/2900057,地震、豪雨夾擊！中橫落石阻斷明搶通 蘇花公路夜間恐預警封閉
1,https://news.tvbs.com.tw/life/2895764,花蓮暴雨頻傳災情！7學生砂婆噹溪「戲水受困」
2,https://news.tvbs.com.tw/politics/2877040,豪雨重創花蓮！徐榛蔚喊話賴清德「不要搭專機」 體驗交通之苦
3,https://news.tvbs.com.tw/life/2877038,3天2中斷！台鐵單線雙向 花端午訂房剩3.5成
4,https://news.tvbs.com.tw/local/2876204,鐵路公路雙癱瘓！台鐵和仁段又豪雨停駛 蘇花崇德段只出不進


In [206]:
# url = 'https://news.tvbs.com.tw/local/2900057'
data['Context'] = ''; data['PublishTime'] = ''

failed_urls = []

for i, url in enumerate(data['Link']):
    print(f'[{i+1}/{len(data)}] 處理中：{url}')

    try:
        resp = req.get(url)

        if resp.status_code != 200:
            print(f'無法存取：{resp.status_code}')
            data.at[i, 'Context'] = ''
            data.at[i, 'PublishTime'] = ''
            failed_urls.append(url)
            continue

        soup = B(resp.text, 'html.parser')

        article_div = soup.find('div', id='news_detail_div') or soup.find('div', class_='article_content')
        context = []
    
        if isinstance(article_div, Tag):
            for node in article_div.descendants:
                if getattr(node, 'get', lambda x: None)('id') == 'article-end-marker':
                    break
                if isinstance(node, NavigableString) and not isinstance(node, (Comment, Doctype)):
                    text = node.strip()
                    if text:
                        context.append(text)

            full_text = '\n'.join(context)
            data.at[i, 'Context'] = full_text
        else:
            print('⚠️ 找不到 <div class="article_content">')
            data.at[i, 'Context'] = ''
            failed_urls.append(url)

        author_div = soup.find('div', class_='author')
        pub_time = ''

        if author_div:
            text = author_div.get_text(separator='\n', strip=True)
            pub_line = next((line for line in text.split('\n') if '發佈時間' in line), '')
            pub_time = pub_line.replace('發佈時間：', '').strip()
        else:
            pub_node = soup.find(string=lambda t: '發佈時間' in t)
            if pub_node:
                pub_time = pub_node.replace('發佈時間：', '').strip()

        data.at[i, 'PublishTime'] = pub_time

    except Exception as e:
        print(f'發生錯誤:{e}')
        data.at[i, 'Context'] = ''
        data.at[i, 'PublishTime'] = ''
        failed_urls.append(url)

    data.to_csv('tvbs_temp.csv', index=False)
    time.sleep(1) 

data.to_csv('tvbs_蘇花公路_full.csv', index=False)
with open('tvbs_蘇花公路_failed.txt', 'w', encoding='utf-8') as f:
    for url in failed_urls:
        f.write(url+'\n')
print('全部完成, 已儲存完整資料和失敗網址')

[1/1183] 處理中：https://news.tvbs.com.tw/local/2900057
[2/1183] 處理中：https://news.tvbs.com.tw/life/2895764
[3/1183] 處理中：https://news.tvbs.com.tw/politics/2877040
[4/1183] 處理中：https://news.tvbs.com.tw/life/2877038
[5/1183] 處理中：https://news.tvbs.com.tw/local/2876204
[6/1183] 處理中：https://news.tvbs.com.tw/life/2868397
[7/1183] 處理中：https://news.tvbs.com.tw/life/2826109
[8/1183] 處理中：https://news.tvbs.com.tw/local/2738907
[9/1183] 處理中：https://news.tvbs.com.tw/local/2705030
[10/1183] 處理中：https://news.tvbs.com.tw/life/2672707
[11/1183] 處理中：https://news.tvbs.com.tw/life/2672462
[12/1183] 處理中：https://news.tvbs.com.tw/life/2672106
[13/1183] 處理中：https://news.tvbs.com.tw/life/2671312
[14/1183] 處理中：https://news.tvbs.com.tw/life/2671211
[15/1183] 處理中：https://news.tvbs.com.tw/life/2671058
[16/1183] 處理中：https://news.tvbs.com.tw/life/2670167
[17/1183] 處理中：https://news.tvbs.com.tw/life/2638802
[18/1183] 處理中：https://news.tvbs.com.tw/life/2637848
[19/1183] 處理中：https://news.tvbs.com.tw/life/2636776
[20/1183] 處理中

In [207]:
df = pd.read_csv('tvbs_蘇花公路_full.csv')
df

,Link,Title,Context,PublishTime
0,https://news.tvbs.com.tw/local/2900057,地震、豪雨夾擊！中橫落石阻斷明搶通 蘇花公路夜間恐預警封閉,受到昨（11）日東部規模6.4地震及熱帶低壓降雨影響，中橫公路台8線157.7k豁然亭路段，...,2025/06/12 15:50
1,https://news.tvbs.com.tw/life/2895764,花蓮暴雨頻傳災情！7學生砂婆噹溪「戲水受困」,花蓮地區下午到晚上期間，大雨與雷聲不間斷，有7名學生到知名景點砂婆噹戲水，返程時，遇到溪水暴...,2025/06/08 20:53
2,https://news.tvbs.com.tw/politics/2877040,豪雨重創花蓮！徐榛蔚喊話賴清德「不要搭專機」 體驗交通之苦,近日豪雨影響，花蓮北上鐵公路再度中斷，導致交通受阻，縣長徐榛蔚今天攜手多位議員及觀光產業界，...,2025/05/21 13:10
3,https://news.tvbs.com.tw/life/2877038,3天2中斷！台鐵單線雙向 花端午訂房剩3.5成,台鐵北迴線，受到大雨影響，兩度雙向中斷無法通行，昨天連蘇花公路都受影響，台鐵宣布在今天清晨4...,2025/05/21 12:49
4,https://news.tvbs.com.tw/local/2876204,鐵路公路雙癱瘓！台鐵和仁段又豪雨停駛 蘇花崇德段只出不進,花蓮縣18日下午受到對流雲系發展旺盛影響，下起傾盆大雨，導致台鐵「和仁=崇德」遭土石流淹沒軌...,2025/05/20 17:11
...,...,...,...,...
1178,https://news.tvbs.com.tw/life/392462,豪雨不斷蘇花坍方 中午恢復通車,受到豪雨影響，宜蘭蘇花公路140.6公里南澳路段，上午嚴重坍方，工務單位搶救三個半小時，中午...,2003/11/28 16:17
1179,https://news.tvbs.com.tw/life/392512,蘇花公路140.6公里處 坍方30公尺,上午七點多，蘇花公路140.6公里處，因為連夜大雨，坍方30公尺，造成雙向全線中斷，許多車輛...,2003/11/28 11:15
1180,https://news.tvbs.com.tw/life/395482,蘇花隧道機車撞卡車 2騎士死亡,台九線蘇花公路崇德隧道口，深夜一輛機車疑似超速失控，迎面撞上對向行駛的一部砂石車，肇事的兩名...,2003/11/04 07:20
1181,https://news.tvbs.com.tw/life/408787,蘇花和仁164公里處坍方 單線通車,蘇花公路又傳坍方，在和仁段164公里處，因為巨石砸毀道路，工程單位緊急搶修，目前單線通車，預...,2003/07/11 13:01


In [208]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1183 entries, 0 to 1182
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Link         1183 non-null   object
 1   Title        1183 non-null   object
 2   Context      1177 non-null   object
 3   PublishTime  1178 non-null   object
dtypes: object(4)
memory usage: 37.1+ KB
